# Diabetes readmission: cleaning and preprocessing

**Question:** At discharge, which patients in this historical dataset were recorded as readmitted within 30 days?

The source is the [UCI Diabetes 130-US Hospitals dataset](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008). Run this notebook from the project folder. It reads the original CSVs without modifying them. Each row is an encounter; `patient_nbr` identifies a person who may have several encounters.

This notebook audits the data, defines the eligible cohort, creates a binary target, separates patients into train/test groups, and prepares features using training data only. The resulting arrays are for a later modelling notebook. The data come from 1999–2008, so any result is a historical learning exercise.

In [ ]:
from pathlib import Path
import csv

import numpy as np
import pandas as pd

data_dir = Path.cwd()
if not (data_dir / "diabetic_data.csv").exists():
    data_dir = data_dir.parent  # also works if the notebook is moved into a notebooks/ folder

raw = pd.read_csv(data_dir / "diabetic_data.csv", keep_default_na=False)
print(f"Rows: {len(raw):,}; columns: {raw.shape[1]}")
print(f"Unique patients: {raw['patient_nbr'].nunique():,}")
print(f"Unique encounter IDs: {raw['encounter_id'].nunique():,}")
print("\nReadmission labels:")
print(raw["readmitted"].value_counts().to_string())

## 1. Audit missing markers and unusual values

`keep_default_na=False` preserves the literal `None` category in lab-result fields. In this CSV, `?` is the explicit missing marker. The separate `IDS_mapping.csv` has three small code tables separated by blank rows; it is not one rectangular dataset.

In [ ]:
question_marks = raw.eq("?").sum()
missing_audit = (
    pd.DataFrame({
        "question_marks": question_marks,
        "percent_of_rows": 100 * question_marks / len(raw),
        "distinct_values": raw.nunique(),
    })
    .loc[lambda d: d["question_marks"] > 0]
    .sort_values("question_marks", ascending=False)
)
print(missing_audit.round(1).to_string())
print("\nSingle-value columns:", raw.columns[raw.nunique() == 1].tolist())
print("Duplicate encounter IDs:", raw["encounter_id"].duplicated().sum())
print("Exact duplicate rows:", raw.duplicated().sum())
print("Patients with multiple encounters:", (raw["patient_nbr"].value_counts() > 1).sum())

stay_days = pd.to_numeric(raw["time_in_hospital"], errors="coerce")
invalid_stays = raw["time_in_hospital"].notna() & stay_days.isna()
if invalid_stays.any():
    print("Nonnumeric length-of-stay values:", int(invalid_stays.sum()))
    print(raw.loc[invalid_stays, "time_in_hospital"].value_counts().head().to_string())
if stay_days.notna().any():
    print("Length-of-stay range:", int(stay_days.min()), "to", int(stay_days.max()), "days")
else:
    print("No valid numeric length-of-stay values")
print("Unknown/Invalid gender rows:", raw["gender"].eq("Unknown/Invalid").sum())

In [ ]:
code_columns = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
code_labels = {}
current_column = None

with (data_dir / "IDS_mapping.csv").open(newline="", encoding="utf-8-sig") as file:
    for row in csv.reader(file):
        key = row[0].strip() if row else ""
        if key in code_columns:
            current_column = key
            code_labels[current_column] = {}
        elif key.isdigit() and current_column is not None:
            code_labels[current_column][int(key)] = row[1].strip()

for column in code_columns:
    print(f"\n{column}:")
    counts = raw[column].value_counts().head(8)
    for code_value, count in counts.items():
        print(f"  {code_value:>2}  {code_labels[column].get(code_value, 'Unmapped')}  ({count:,})")

## 2. Clean and define the cohort

These rules are explicit so they can be revisited:

- Replace `?` with missing values. Keep `None` as a real category meaning no recorded test result.
- Treat `Unknown/Invalid` gender as missing. Parse numeric stay and count fields as numbers; report invalid text and treat it as missing.
- Exclude encounters ending in death (discharge codes 11, 19, 20, 21): these patients are not eligible for a later readmission at discharge. The mapping file supplies the code descriptions.
- Define `readmitted_30d = 1` only for `<30`; `>30` and `NO` are 0. `NO` means no readmission recorded in this dataset, which is narrower than proving none occurred anywhere.
- Keep all original columns in `cohort` for audit. Drop identifiers, the label, `weight` (96.9% missing), and the two single-value medication fields only from the model feature table. The missingness decision is a simple fixed rule for this exercise; we can test alternatives later.

In [ ]:
assert raw["encounter_id"].is_unique
assert set(raw["readmitted"].unique()) == {"<30", ">30", "NO"}

clean = raw.replace("?", pd.NA)
clean["gender"] = clean["gender"].replace("Unknown/Invalid", pd.NA)

numeric_feature_columns = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]
for column in numeric_feature_columns:
    parsed = pd.to_numeric(clean[column], errors="coerce")
    invalid = clean[column].notna() & parsed.isna()
    if invalid.any():
        print(f"Invalid numeric values in {column}: {int(invalid.sum())}")
        print(clean.loc[invalid, column].value_counts().head().to_string())
    clean[column] = parsed

expired_codes = {11, 19, 20, 21}
expired_mask = clean["discharge_disposition_id"].isin(expired_codes)
print("Encounters ending in death:", int(expired_mask.sum()))
print("Their recorded labels:", clean.loc[expired_mask, "readmitted"].value_counts().to_dict())

cohort = clean.loc[~expired_mask].copy()
cohort["readmitted_30d"] = cohort["readmitted"].eq("<30").astype("int8")

excluded_features = [
    "encounter_id", "patient_nbr", "readmitted", "readmitted_30d",
    "weight", "examide", "citoglipton",
]
features = cohort.drop(columns=excluded_features).copy()
for column in code_columns:
    features[column] = features[column].astype("string")  # codes are categories, not quantities

print(f"Eligible encounters: {len(cohort):,}")
print(f"Model features: {features.shape[1]}")
print(f"30-day readmission rate: {cohort['readmitted_30d'].mean():.2%}")

## 3. Split by patient before learning preprocessing values

The same patient must not appear in both groups. The random seed makes the split reproducible. This is a random patient split; the CSV has no encounter date for a later-time validation. Do not use `patient_nbr` or `encounter_id` as model inputs.

In [ ]:
rng = np.random.default_rng(42)
patient_ids = rng.permutation(cohort["patient_nbr"].drop_duplicates().to_numpy())
train_cutoff = int(0.8 * len(patient_ids))
train_patient_ids = set(patient_ids[:train_cutoff])

train_mask = cohort["patient_nbr"].isin(train_patient_ids)
X_train = features.loc[train_mask].copy()
X_test = features.loc[~train_mask].copy()
y_train = cohort.loc[train_mask, "readmitted_30d"].copy()
y_test = cohort.loc[~train_mask, "readmitted_30d"].copy()

assert set(cohort.loc[train_mask, "patient_nbr"]).isdisjoint(
    set(cohort.loc[~train_mask, "patient_nbr"])
)
assert len(X_train) + len(X_test) == len(cohort)
print(f"Train: {len(X_train):,} encounters, {y_train.mean():.2%} positive")
print(f"Test:  {len(X_test):,} encounters, {y_test.mean():.2%} positive")
print(f"Train patients: {train_cutoff:,}; test patients: {len(patient_ids) - train_cutoff:,}")

## 4. Fit simple preprocessing on training data only

Numeric missing values use training medians. Categorical missing values become an explicit `Missing` category. One-hot columns are learned from training data; unseen test categories cannot create new columns. Sparse dummy columns keep the wide diagnosis-code representation manageable. The original `X_train` and `X_test` remain available for later experiments. A later model pipeline can replace these simple rules.

In [ ]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = [column for column in X_train if column not in numeric_columns]

train_medians = X_train[numeric_columns].median()
train_ready = X_train.copy()
test_ready = X_test.copy()
train_ready[numeric_columns] = train_ready[numeric_columns].fillna(train_medians)
test_ready[numeric_columns] = test_ready[numeric_columns].fillna(train_medians)
train_ready[categorical_columns] = train_ready[categorical_columns].fillna("Missing")
test_ready[categorical_columns] = test_ready[categorical_columns].fillna("Missing")

X_train_prepared = pd.get_dummies(
    train_ready, columns=categorical_columns, sparse=True, dtype=np.uint8
)
X_test_prepared = pd.get_dummies(
    test_ready, columns=categorical_columns, sparse=True, dtype=np.uint8
).reindex(columns=X_train_prepared.columns, fill_value=0)

assert X_train_prepared.columns.equals(X_test_prepared.columns)
assert not X_train_prepared.columns.duplicated().any()
print(f"Numeric features: {len(numeric_columns)}")
print(f"Categorical features before encoding: {len(categorical_columns)}")
print(f"Prepared train shape: {X_train_prepared.shape}")
print(f"Prepared test shape:  {X_test_prepared.shape}")
print("Training-only median values:", train_medians.to_dict())

The prepared data are held in memory; the source CSVs remain unchanged. The next stage is to compare a constant baseline and logistic regression on this fixed split, then inspect recall, precision, calibration, and performance across patient groups. Do not interpret these historical offline results as current clinical performance.